# The shared finance corpus

Every finance case study on formalNLP argues from **one** corpus, generated by `finance_corpus.py`.
It is built in two layers: a fixed **world** (a knowledge base about a fictional roster of companies)
and **text** rendered from that world. This notebook tours both and then runs the verification harness
that downstream topics rely on. Nothing here redefines the generator: every number comes from the module.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd()))
import finance_corpus as fc

docs = fc.generate_corpus(300, seed=0)
tokens = [t for d in docs for t in fc._tokens(d.text)]
print(len(docs), "documents,", len(tokens), "tokens,", len(set(tokens)), "types")
print("tones:", Counter(d.tone for d in docs))
print("doc types:", Counter(d.doc_type for d in docs))

## The world comes first

`fc.WORLD` is built from `WORLD_SEED` alone, so every corpus, whatever its seed, describes the same
world. Its relations have the structure knowledge-graph embedding topics need:

* `COMPETES_WITH` is **symmetric**;
* `SUBSIDIARY_OF` and `PARENT_OF` are **inverses**;
* `BASED_IN` is the **composition** `HQ_IN ∘ LOCATED_IN` (company → city → region);
* every `ACQUIRED` target is a subsidiary acquired exactly once, so `ACQUIRED ⇒ PARENT_OF`.

In [ ]:
kb = fc.WORLD
print(Counter(kb.types.values()))
print(Counter(r for _, r, _ in kb.triples))
h, t = kb.facts("COMPETES_WITH")[0]
print("symmetric:", (h, t), "and", (t, h), "both in KB:", (t, "COMPETES_WITH", h) in kb.triples)
sub, parent = kb.facts("SUBSIDIARY_OF")[0]
print("inverse:", sub, "SUBSIDIARY_OF", parent, "<=>", parent, "PARENT_OF", sub)
org, city = kb.facts("HQ_IN")[0]
print("composition:", org, "HQ_IN", city, "LOCATED_IN", fc.CITIES[city], "=> BASED_IN", kb.between(org, fc.CITIES[city]))

## One document, with its gold annotations

Documents render sentences from those facts, with several paraphrases per relation, and add
**distractor** sentences that co-mention a pair without stating any relation. Gold annotations:

* entity spans (`ORG`, `PERSON`, `GPE`, `SECTOR`, `MONEY`, `PERCENT`, `DATE`, `PRODUCT`), with `kb_id`
  linking names and aliases to the world;
* relations the text **states**, plus their logical equivalents (the symmetric twin, the inverse);
* aspect terms with a polarity, and gold sentence spans.

Span conventions: `PRODUCT` and `DATE` spans include a leading "the" when the name takes one ("the
Kestrel turbine", "the third quarter"). `ORG`, `PERSON`, `GPE`, and `SECTOR` spans never do.

In [ ]:
d = docs[3]
print(d.doc_id, d.company, d.doc_type, "tone =", d.tone, "\n")
for s, e in d.sentences[:8]:
    print("  ", d.text[s:e])
print()
for r in d.relations:
    head, tail = d.entities[r.head], d.entities[r.tail]
    print(f"{d.text[head.start:head.end]!r:32} {r.rel:14} {d.text[tail.start:tail.end]!r}")
print()
for a in d.aspects:
    print(f"{d.text[a.start:a.end]!r:22} {a.category:12} {a.polarity:+d}")

## An open vocabulary: Zipf and Heaps

Content words are drawn from Pitman–Yor processes (one per sector, one shared) with discount
$d = $ `HEAPS_DISCOUNT`. In the Chinese-restaurant view a new word type appears with probability
$(\theta + dK)/(\theta + n)$, which makes $\mathbb{E}[K_n] + \theta/d \propto \Gamma(\theta+n+d)/\Gamma(\theta+n) \approx (\theta+n)^d$:
**Heaps' law, with exponent equal to the discount**. Past the seeded real nouns, new types are
pseudo-words built from English-like morphemes ("subledgers", "freightbook").

The template words around them (*the*, *in*, *was*) form a closed class that saturates early, so the
full-text Heaps exponent sits below $d$. Function words behave the same way in real text. The
rank–frequency slope over the top 200 ranks is the template head; over log-spaced ranks 1–1000 it is
close to Zipf's $-1$.

In [ ]:
counts = Counter(tokens)
head = np.array(sorted(counts.values(), reverse=True)[:200], dtype=float)
print(f"Zipf slope, log-spaced ranks 1–1000: {fc.zipf_slope(tokens):.3f}")
print(f"Zipf slope, ranks 1–200 (template head): {np.polyfit(np.log(np.arange(1, 201)), np.log(head), 1)[0]:.3f}")
print(f"Heaps exponent, full text: {fc.heaps_exponent(tokens):.3f}   (configured discount d = {fc.HEAPS_DISCOUNT})")
print("hapax legomena:", sum(c == 1 for c in counts.values()), "of", len(counts), "types")
for disc in (0.3, 0.6, 0.8):
    toks = [t for x in fc.generate_corpus(150, 0, discount=disc) for t in fc._tokens(x.text)]
    print(f"  discount {disc}: {len(set(toks)):5d} types in {len(toks)} tokens")

Unseen events are what smoothing is for. Bigrams seen once, and words in held-out documents that never
appeared in training, are the raw material of Good–Turing, Kneser–Ney, and perplexity.

In [ ]:
bigrams = Counter(p for x in docs for s, e in x.sentences
                  for p in zip(fc._tokens(x.text[s:e]), fc._tokens(x.text[s:e])[1:]))
print(f"singleton share of bigram types: {sum(c == 1 for c in bigrams.values()) / len(bigrams):.3f}")
order = np.random.default_rng(0).permutation(len(docs))
train = {t for i in order[:240] for t in fc._tokens(docs[i].text)}
held = [t for i in order[240:] for t in fc._tokens(docs[i].text)]
print(f"held-out OOV token rate: {np.mean([t not in train for t in held]):.3f}")

## Distant supervision is noisy by construction

Distant supervision labels every sentence that co-mentions a related pair with the KB relation. Here the
generator knows which sentences actually state the fact, so the noise rate is measurable rather than
assumed. Distractors, unstated inverses, and the composed `BASED_IN` behind "headquartered in Toronto,
Canada" all count as false positives.

In [ ]:
print(f"distant-supervision precision: {fc.distant_supervision_precision(docs):.3f}")
stated = {(x.entities[r.head].kb_id, r.rel, x.entities[r.tail].kb_id) for x in docs for r in x.relations}
print(f"KB facts stated at least once: {len(stated & kb.triples) / len(kb.triples):.3f}")

## Held-out names for NER

`split_roster` holds out one company per sector. A corpus generated with `roster=train` never
mentions a held-out company, its subsidiaries, or its officers, so a tagger evaluated on
`roster=held` has to generalize to names it has never seen.

In [ ]:
train_roster, held_roster = fc.split_roster()
print("held out:", held_roster)
train_docs = fc.generate_corpus(200, 1, roster=train_roster)
held_docs = fc.generate_corpus(100, 2, roster=held_roster)
seen = {x.text[e.start:e.end] for x in train_docs for e in x.entities if e.type in ("ORG", "PERSON")}
unseen = {x.text[e.start:e.end] for x in held_docs for e in x.entities if e.type in ("ORG", "PERSON")}
print(f"{len(unseen)} held-out ORG/PERSON surface forms, {len(unseen & seen)} of them seen in training")

## Sentiment: lexicons, aspects, and polysemes

`STANDIN_LEXICON` is hand-written for this site. It is **not** the Loughran–McDonald dictionary, whose
license does not allow redistribution here. Its net score
$\text{score}(x) = (\#\text{pos} - \#\text{neg}) / \max(1, \#\text{tokens})$ tracks the gold tone, but
only moderately. Documents mix aspect polarities, hedge in every class, and carry polarity in ways a
bag of words cannot see:

* **direction × valence**: "costs rose" is bad and "costs fell" is good, while *rose* and *fell* carry
  no listed polarity;
* **valence traps**: "costs exceeded expectations" uses a positive word for bad news;
* **negation**: "was not strong".

`GENERAL_STANDIN_LEXICON` adds the finance polysemes (*liability*, *tax*, *cost*, *crude*, *capital*)
that a general-purpose lexicon counts as negative. That misreading costs it correlation, as Loughran and
McDonald found for the Harvard-IV list.

In [ ]:
tones = [x.tone for x in docs]
rho = spearmanr([fc.lexicon_score(x.text) for x in docs], tones).statistic
rho_gen = spearmanr([fc.lexicon_score(x.text, fc.GENERAL_STANDIN_LEXICON) for x in docs], tones).statistic
print(f"Spearman(finance lexicon, tone) = {rho:.3f}")
print(f"Spearman(general lexicon, tone) = {rho_gen:.3f}")
print(f"three-line lexicon rule accuracy = {np.mean([fc.lexicon_rule(x.text) == x.tone for x in docs]):.3f}")
print("aspect polarity by category:")
for cat in fc.ASPECT_CATEGORIES:
    print(f"  {cat:12} {dict(sorted(Counter(a.polarity for x in docs for a in x.aspects if a.category == cat).items()))}")

`label_noise` flips each tone label with the given probability from a separate random stream, so the text
is byte-identical and `clean_tone` keeps the truth. This supports noisy-label experiments.

In [ ]:
noisy = fc.generate_corpus(300, 0, label_noise=0.2)
print("texts identical:", [x.text for x in noisy] == [x.text for x in docs])
print(f"flipped: {np.mean([x.tone != x.clean_tone for x in noisy]):.3f}")

## Public EDGAR excerpts

These are real 10-K MD&A text and **unlabeled**: `tone is None`, which is distinct from a labeled-neutral
`tone == 0`.

In [ ]:
edgar = fc.load_edgar_excerpts()
print(len(edgar), "excerpts")
for e in edgar:
    print(f"- {e.doc_id} ({e.company}): {len(e.text.split())} words, tone = {e.tone}")

## Verification harness

In [ ]:
assert fc._run_harness() == 0